In [1]:
import os

os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"

import pandas as pd
import numpy as np
import re
from tqdm import tqdm
import warnings

warnings.simplefilter("ignore")



In [2]:
PATH = "../input/google-quest-challenge/"

df_train = pd.read_csv(PATH + "train.csv")
df_test = pd.read_csv(PATH + "test.csv")
df_sub = pd.read_csv(PATH + "sample_submission.csv")

output_categories = list(df_train.columns[11:])
print("\nOutput Categories:\n", output_categories)




Output Categories:
 ['question_asker_intent_understanding', 'question_body_critical', 'question_conversational', 'question_expect_short_answer', 'question_fact_seeking', 'question_has_commonly_accepted_answer', 'question_interestingness_others', 'question_interestingness_self', 'question_multi_intent', 'question_not_really_a_question', 'question_opinion_seeking', 'question_type_choice', 'question_type_compare', 'question_type_consequence', 'question_type_definition', 'question_type_entity', 'question_type_instructions', 'question_type_procedure', 'question_type_reason_explanation', 'question_type_spelling', 'question_well_written', 'answer_helpful', 'answer_level_of_information', 'answer_plausible', 'answer_relevance', 'answer_satisfaction', 'answer_type_instructions', 'answer_type_procedure', 'answer_type_reason_explanation', 'answer_well_written']


In [3]:
stopwords = [
    "i",
    "me",
    "my",
    "myself",
    "we",
    "our",
    "ours",
    "ourselves",
    "you",
    "you're",
    "you've",
    "you'll",
    "you'd",
    "your",
    "yours",
    "yourself",
    "yourselves",
    "he",
    "him",
    "his",
    "himself",
    "she",
    "she's",
    "her",
    "hers",
    "herself",
    "it",
    "it's",
    "its",
    "itself",
    "they",
    "them",
    "their",
    "theirs",
    "themselves",
    "what",
    "which",
    "who",
    "whom",
    "this",
    "that",
    "that'll",
    "these",
    "those",
    "am",
    "is",
    "are",
    "was",
    "were",
    "be",
    "been",
    "being",
    "have",
    "has",
    "had",
    "having",
    "do",
    "does",
    "did",
    "doing",
    "a",
    "an",
    "the",
    "and",
    "but",
    "if",
    "or",
    "because",
    "as",
    "until",
    "while",
    "of",
    "at",
    "by",
    "for",
    "with",
    "about",
    "against",
    "between",
    "into",
    "through",
    "during",
    "before",
    "after",
    "above",
    "below",
    "to",
    "from",
    "up",
    "down",
    "in",
    "out",
    "on",
    "off",
    "over",
    "under",
    "again",
    "further",
    "then",
    "once",
    "here",
    "there",
    "when",
    "where",
    "why",
    "how",
    "all",
    "any",
    "both",
    "each",
    "few",
    "more",
    "most",
    "other",
    "some",
    "such",
    "only",
    "own",
    "same",
    "so",
    "than",
    "too",
    "very",
    "s",
    "t",
    "can",
    "will",
    "just",
    "don",
    "don't",
    "should",
    "should've",
    "now",
    "d",
    "ll",
    "m",
    "o",
    "re",
    "ve",
    "y",
    "ain",
    "aren",
    "aren't",
    "couldn",
    "couldn't",
    "didn",
    "didn't",
    "doesn",
    "doesn't",
    "hadn",
    "hadn't",
    "hasn",
    "hasn't",
    "haven",
    "haven't",
    "isn",
    "isn't",
    "ma",
    "mightn",
    "mightn't",
    "mustn",
    "mustn't",
    "needn",
    "needn't",
    "shan",
    "shan't",
    "shouldn",
    "shouldn't",
    "wasn",
    "wasn't",
    "weren",
    "weren't",
    "won",
    "won't",
    "wouldn",
    "wouldn't",
]


def decontracted(phrase):
    phrase = re.sub(r"won't", "will not", phrase)
    phrase = re.sub(r"can\'t", "can not", phrase)
    phrase = re.sub(r"n\'t", " not", phrase)
    phrase = re.sub(r"\'re", " are", phrase)
    phrase = re.sub(r"\'s", " is", phrase)
    phrase = re.sub(r"\'d", " would", phrase)
    phrase = re.sub(r"\'ll", " will", phrase)
    phrase = re.sub(r"\'t", " not", phrase)
    phrase = re.sub(r"\'ve", " have", phrase)
    phrase = re.sub(r"\'m", " am", phrase)
    return phrase


def preprocess_text(text_series):
    processed = []
    for sent in tqdm(text_series, leave=False):
        sent = decontracted(str(sent))
        sent = sent.replace("\\r", " ").replace("\\n", " ").replace('\\"', " ")
        sent = re.sub("[^A-Za-z0-9]+", " ", sent)
        sent = " ".join(w for w in sent.split() if w.lower() not in stopwords)
        processed.append(sent.lower().strip())
    return processed


df_train["Preproc_Question_Title"] = preprocess_text(df_train["question_title"])
df_train["Preproc_Question_Body"] = preprocess_text(df_train["question_body"])
df_train["Preproc_Answer"] = preprocess_text(df_train["answer"])

df_test["Preproc_Question_Title"] = preprocess_text(df_test["question_title"])
df_test["Preproc_Question_Body"] = preprocess_text(df_test["question_body"])
df_test["Preproc_Answer"] = preprocess_text(df_test["answer"])



In [4]:
df_train["combined_text"] = (
    df_train["Preproc_Question_Title"]
    + " "
    + df_train["Preproc_Question_Body"]
    + " "
    + df_train["Preproc_Answer"]
)

df_test["combined_text"] = (
    df_test["Preproc_Question_Title"]
    + " "
    + df_test["Preproc_Question_Body"]
    + " "
    + df_test["Preproc_Answer"]
)

from sklearn.feature_extraction.text import TfidfVectorizer

# Increase vocabulary size to give the linear model more discriminative features
vectorizer = TfidfVectorizer(
    max_features=3000,
    ngram_range=(1, 2),
    stop_words="english",
    dtype=np.float32,
)

X_train_full = vectorizer.fit_transform(df_train["combined_text"])
X_test = vectorizer.transform(df_test["combined_text"])



In [5]:
# Use a larger training subset to improve model stability
desired_sample = 20000
sample_size = min(desired_sample, X_train_full.shape[0])
rng = np.random.default_rng(42)
sample_idx = rng.choice(X_train_full.shape[0], size=sample_size, replace=False)

X_train = X_train_full[sample_idx]
y_train = df_train.loc[sample_idx, output_categories].values

from sklearn.linear_model import Ridge

models = {}
for i, col in enumerate(output_categories):
    # Reduce regularisation so predictions are more correlated with targets
    ridge = Ridge(alpha=500.0, solver="lsqr")
    ridge.fit(X_train, y_train[:, i])
    models[col] = ridge
    if (i + 1) % 5 == 0:
        print(f"Trained {i + 1}/{len(output_categories)} models")

preds = np.zeros((X_test.shape[0], len(output_categories)), dtype=np.float32)

for i, col in enumerate(output_categories):
    preds[:, i] = models[col].predict(X_test)

# Remove added noise to keep predictions more aligned with the model output
noise_std = 0.0
if noise_std > 0:
    noise_rng = np.random.default_rng(123)
    preds = preds + noise_rng.normal(0.0, noise_std, preds.shape)

preds = np.clip(preds, 0.0, 1.0)

submission_df = pd.concat(
    [
        df_test[["qa_id"]].reset_index(drop=True),
        pd.DataFrame(preds, columns=output_categories),
    ],
    axis=1,
)

submission_path = "submission.csv"
submission_df.to_csv(submission_path, index=False)
print(f"Submission file written to {submission_path}")

Trained 5/30 models
Trained 10/30 models
Trained 15/30 models
Trained 20/30 models
Trained 25/30 models
Trained 30/30 models
Submission file written to submission.csv
